Часть 1 Пайплайны

Для эмбеддингов (поиска по базе):

cointegrated/rubert-tiny2 (всего 29 Мб, летает на процессоре, отлично понимает русский).

intfloat/multilingual-e5-small (чуть тяжелее, но выдает топовое качество векторного поиска).

Для генерации ответов (LLM):

IlyaGusev/saiga_llama3_8b (отличная русскоязычная Llama-3, потребует квантования до 4/8 бит, чтобы влезть в GPU).

Qwen/Qwen2.5-1.5B-Instruct (крошечная LLM на 1.5 млрд параметров от Alibaba, очень умная для своих размеров, может работать даже на CPU, отлично выдает JSON).

Абстракция высшего уровня: pipeline
Pre-processing: Взять текст 
→ Токенизатор 
→ Тензоры input_ids.

Forward Pass: Засунуть тензоры в модель 
→ Получить сырые логиты или векторы.

Post-processing: Взять логиты 
→ argmax / softmax / декодирование 
→ Читаемый текст.

Когда ты тестируешь гипотезы на хакатоне, писать эти три шага каждый раз вручную — долго. Библиотека transformers предлагает класс pipeline, который заворачивает весь этот процесс в одну строку кода.

Быстрая классификация (Sentiment Analysis)

Допустим, ты хочешь проверить, ругается ли клиент в чате, чтобы сразу перевести его на живого оператора, а не на бота.

In [1]:
from transformers import pipeline

classifier = pipeline(
    task="sentiment-analysis",
    model="blanchefort/rubert-base-cased-sentiment"
)

result = classifier("У меня списали деньги за тариф, хотя я им не пользовался! Ужас!")
print(result)

c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\IVAN\.cache\huggingface\hub\models--blanchefort--rubert-base-cased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run 

[{'label': 'NEGATIVE', 'score': 0.7516347765922546}]


Извлечение фичей (Feature Extraction для RAG)

Если тебе нужно быстро получить эмбеддинги (векторы) текста для сохранения в базу данных без ручной возни с пулингом (усреднением токенов):

In [7]:
from transformers import pipeline

extractor = pipeline(
    task="feature-extraction",
    model="cointegrated/rubert-tiny2",
    device=0 # GPU
)

embeddings = extractor("Тариф МТС Супер включает 50 ГБ интернета")
# список списков, с эмбеддингами
print(len(embeddings), len(embeddings[0]), len(embeddings[0][0]))
print(embeddings)

Device set to use cpu


1 10 312
[[[0.07904082536697388, 0.22120161354541779, 0.043220844119787216, -1.0963764190673828, 0.11530798673629761, 0.4088938236236572, -0.834014356136322, -0.9357492923736572, -0.3096655011177063, 0.27560511231422424, 0.5451028943061829, 0.26417484879493713, 0.6415582299232483, 0.04883549362421036, 1.2667542695999146, 0.07702881842851639, -0.1547008454799652, -0.3099121153354645, 0.05993284285068512, 0.5430777668952942, -0.42305558919906616, -0.24419206380844116, -0.03197326883673668, -0.5535884499549866, 0.21830132603645325, 0.7427298426628113, 0.66279536485672, -0.07986990362405777, -0.38313040137290955, -0.07624892890453339, -0.11225651204586029, 0.3251972198486328, 0.04905795305967331, 0.1428242325782776, -0.08906709402799606, 0.5530788898468018, -0.13163870573043823, 0.45856547355651855, 0.08107756078243256, -0.4555231034755707, -0.969458818435669, 1.8376578092575073, 1.4036396741867065, -0.6188437342643738, -0.5184733867645264, 0.4894927442073822, 0.6154206395149231, 0.4921869

Часть 2 Токенизаторы и Chat Templates

1. Проблема в том, что единого стандарта в индустрии нет. Каждая лаборатория придумывает свои теги:

Формат Llama 3 (Meta): <|start_header_id|>user<|end_header_id|>\nПривет<|eot_id|>

Формат Qwen/Mistral (ChatML): <|im_start|>user\nПривет<|im_end|>

Формат старой Llama 2: [INST] Привет [/INST]

Если ты загрузишь модель Qwen, но попытаешься поговорить с ней, обернув текст в теги от Llama, модель сойдет с ума. Она не распознает теги, решит, что это какой-то программный код, и выдаст галлюцинацию.

Писать руками эти склейки строк (f"<|im_start|>{role}\n{text}<|im_end|>") на хакатоне — это путь к багам и боли, особенно когда диалог становится длинным.

2. Решение: apply_chat_template

Инженеры Hugging Face элегантно решили эту проблему. Поскольку формат диалога жестко привязан к конкретной модели, они вшили этот формат прямо в токенизатор этой модели.

Тебе больше не нужно знать, какие спец-теги использует выбранная тобой LLM. Ты общаешься с токенизатором через универсальный стандарт — список словарей (messages). Этот же стандарт использует OpenAI в своем API.

Как это выглядит в коде для моего ИИ-консультанта МТС(хаккатон)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# 1. Формируем универсальную структуру диалога
messages = [
    # System Prompt - главная инструкция, задающая личность и правила поведения
    {
        "role": "system",
        "content": "Ты — ИИ-консультант МТС EnergyTool. Отвечай кратко, вежливо и предлагай тарифы из контекста."
    },

    # Реплика клиента
    {
        "role": "user",
        "content": "У меня небольшое кафе, плачу за свет космические суммы. Что посоветуешь?"
    }
]

# 2. Магия Chat Template
# Токенизатор сам расставит нужные теги <|im_start|> и <|im_end|> и переведет всё в тензоры
inputs_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,                  # Сразу перевести в числа (тензоры)
    add_generation_prompt=True,     # КРИТИЧЕСКИ ВАЖНО! Читать ниже.
    return_tensors="pt"             # Вернуть тензоры PyTorch
)

3. Зачем нужен add_generation_prompt=True?

Это самая частая ошибка новичков на хакатонах.
Если ты передашь add_generation_prompt=False, токенизатор просто склеит твой список в строку:... <|im_start|>user\nЧто посоветуешь?<|im_end|>

Если подать это в модель, она может начать генерировать мысли от лица пользователя (просто продолжит его реплику).

Флаг add_generation_prompt=True приклеивает в самый конец текста открывающий тег ассистента:... <|im_start|>user\nЧто посоветуешь?<|im_end|>\n<|im_start|>assistant\n`

Получив такой хвост, LLM однозначно понимает: "Ага, реплика пользователя закончилась, мне передали микрофон, теперь я должен генерировать ответ Ассистента".

4. Декодирование (Очистка от служебного мусора)

Когда LLM сгенерирует ответ, в нем будут присутствовать те самые закрывающие токены (например, <|im_end|>). Если ты отдашь этот сырой текст фронтендеру, клиент увидит на сайте МТС страшные системные теги.

Чтобы этого избежать, при расшифровке ответа (инференсе) всегда используй флаг skip_special_tokens:

In [ ]:
# Декодируем только новые сгенерированные токены (без самого промпта)
# generated_ids - это то, что выдаст метод model.generate()
clean_text = tokenizer.decode(
    generated_ids[0], 
    skip_special_tokens=True
)

1. Магия под капотом: Apache Arrow

В стандартном Python, если ты загрузишь в память массив текстов на 20 ГБ, твоя оперативная память (RAM) забьется, и сервер упадет с ошибкой Out of Memory.
Библиотека datasets построена на движке Apache Arrow. Он хранит данные на жестком диске в специальном бинарном формате и подгружает их в оперативную память микроскопическими порциями (Zero-copy reads) только в тот момент, когда они реально нужны процессору.

Даже если организаторы МТС вывалят на тебя 100 гигабайт логов и документации, datasets обработает их на обычном ноутбуке, почти не потребляя RAM.

2. Загрузка локальных данных для RAG

В классическом RAG всё начинается с сырых документов. Допустим, бэкендер спарсил сайт EnergyTool и отдал тебе CSV-файл со столбцами title (название услуги) и content (описание).

Загрузка в формат Hugging Face делается одной строкой:

In [ ]:
from datasets import load_dataset

# Загружаем локальный CSV (можно JSON, TXT, Parquet)
# datasets сам поймет структуру и создаст объект DatasetDict
dataset = load_dataset("csv", data_files="mts_energy_tariffs.csv", split="train")

print(dataset[0])
# Выведет: {'title': 'Тариф Умный Свет', 'content': 'Снижает затраты на...'}

3. Метод .map() — Сердце конвейера

Главный инструмент библиотеки — метод .map(). Он позволяет применить любую твою функцию ко всем строкам датасета. В контексте RAG тебе нужно будет нарезать длинные тексты на куски (чанки) и превратить их в векторы (эмбеддинги).

Делать это циклом for — невероятно долго. Метод .map() умеет распараллеливать вычисления и обрабатывать данные батчами.

In [ ]:
from transformers import pipeline

# Поднимаем легковесную модель для векторизации
embedder = pipeline(
    "feature-extraction", 
    model="cointegrated/rubert-tiny2",
    device=0
)

# Пишем функцию, которая принимает батч текстов и возвращает батч векторов
def get_embeddings(batch):
    # embedder возвращает тензоры, берем вектор токена [CLS] (индекс 0) для каждого текста
    vectors = [out[0][0] for out in embedder(batch["content"])]
    return {"embeddings": vectors}

# Магия параллельной обработки:
# batched=True берёт данные кусками по 1000 строк
# В датасете автоматически появится новая колонка "embeddings"
vectorized_dataset = dataset.map(
    get_embeddings, 
    batched=True, 
    batch_size=128
)

4. Встроенный векторный поиск (Киллер-фича для хакатона)

Самое интересное: библиотека datasets настолько тесно интегрирована с экосистемой ML, что тебе даже не обязательно поднимать отдельную векторную базу данных (вроде ChromaDB или Qdrant) для создания первого прототипа RAG.

В datasets встроена поддержка FAISS (библиотеки от Facebook для молниеносного поиска по векторам)

In [ ]:
# Говорим датасету построить векторный индекс по колонке "embeddings"
vectorized_dataset.add_faiss_index(column="embeddings")

# Теперь, когда клиент пишет запрос в чат:
user_query = "Как сэкономить на освящении склада?"
query_embedding = embedder(user_query)[0][0][0] # получаем вектор запроса

# Ищем топ-3 самых подходящих тарифа МТС прямо внутри датасета!
scores, retrieved_examples = vectorized_dataset.get_nearest_examples(
    "embeddings",
    query_embedding,
    k=3
)

print("Найдено решение:", retrieved_examples["title"][0])